## Structured Outputs
 When we want output in a specific Structure 

### Pydantic

In [1]:
import os
from langchain.chat_models import init_chat_model
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

model=init_chat_model(model="gpt-4.1")
model

ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x1125da1b0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x112fbe2a0>, root_client=<openai.OpenAI object at 0x11252fa70>, root_async_client=<openai.AsyncOpenAI object at 0x112eedfd0>, model_name='gpt-4.1', model_kwargs={}, openai_api_key=SecretStr('**********'))

In [2]:
from pydantic import BaseModel,Field
class Movie(BaseModel):
    title:str=Field(description="Title of the movie")
    year:int=Field(description="This is the year movie got released")
    director:str=Field(description="Director of the movie")
    ratings:float=Field(description="The IMD movie rating")


In [3]:
model_with_structure=model.with_structured_output(Movie)
model_with_structure

RunnableBinding(bound=ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x1125da1b0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x112fbe2a0>, root_client=<openai.OpenAI object at 0x11252fa70>, root_async_client=<openai.AsyncOpenAI object at 0x112eedfd0>, model_name='gpt-4.1', model_kwargs={}, openai_api_key=SecretStr('**********')), kwargs={'response_format': <class '__main__.Movie'>, 'ls_structured_output_format': {'kwargs': {'method': 'json_schema', 'strict': None}, 'schema': {'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'Title of the movie', 'type': 'string'}, 'year': {'description': 'This is the year movie got released', 'type': 'integer'}, 'director': {'description': 'Director of the movie', 'type': 'string'}, 'ratings': {'description': 'The IMD movie rating', 'type': 'number'}}, 'required': ['title', 'year', 'director', '

In [4]:
model_with_structure.invoke("provide details of inception")

Movie(title='Inception', year=2010, director='Christopher Nolan', ratings=8.8)

### Nested Structure

In [6]:
from pydantic import BaseModel, Field
class Actor(BaseModel):
    name:str
    role:str

class MovieDetails(BaseModel):
    title:str
    year:int
    cast:list[Actor]
    genres:list[str]
    budget:float | None=Field(None,description="Budget of movie in dollar($)")

In [7]:
model_with_structure=model.with_structured_output(MovieDetails)
response =model_with_structure.invoke("provide details of inception")
print(response)

title='Inception' year=2010 cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Ellen Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames'), Actor(name='Ken Watanabe', role='Saito'), Actor(name='Cillian Murphy', role='Robert Fischer'), Actor(name='Marion Cotillard', role='Mal Cobb'), Actor(name='Michael Caine', role='Miles')] genres=['Science Fiction', 'Action', 'Thriller'] budget=160000000.0


### Typed Dict

provides simple alternative using python's built in, ideal when you don't need runtime validations

In [13]:
from typing_extensions import TypedDict,Annotated
class MovieDict(TypedDict):
    """ A movie withvDetails """
    title: Annotated[str,..., "The title of the movie"]
    year: Annotated[int, ...,"The year the movie was released"]
    director: Annotated[str,..., "The director of the movie"]
    rating: Annotated[float,...,"The movie's rating out of 10"]

model_with_typDict=model.with_structured_output(MovieDict)
response=model_with_typDict.invoke("give details about movie inception")
response

{'title': 'Inception',
 'year': 2010,
 'director': 'Christopher Nolan',
 'rating': 8.8}

### Data Class

In [8]:
from langchain.agents import create_agent

ImportError: cannot import name 'create_agent' from 'langchain.agents' (/Users/hemantaryapanwar/Developer/Dev/AI Agent/agents/.venv/lib/python3.12/site-packages/langchain/agents/__init__.py)

In [ ]:
from dataclasses import dataclass
from langchain_openai import ChatOpenAI

@dataclass
class ContactInfo:
    """Contact information of a person"""
    name: str
    email: str
    phone: str 

# Create the LLM and bind your schema
llm = ChatOpenAI(model="gpt-4.1")
extractor = llm.with_structured_output(ContactInfo)

# Invoke directly — no agent needed
result = extractor.invoke(
    "Extract info from: john joe, John@example.com, (+55)(9898787890)"
)

print(result)
# ContactInfo(name='John Joe', email='John@example.com', phone='(+55)(9898787890)')

{'name': 'john joe', 'email': 'John@example.com', 'phone': '(+55)(9898787890)'}
